In [ ]:
# GPU and auto-reload
# (In the Colab UI: Runtime ▷ Change runtime type ▷ GPU)
%load_ext autoreload
%autoreload 2


In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/dlfa_capstone'
DATA_ROOT  = f'{DRIVE_ROOT}/datasets/meld_data'
CKPT_ROOT  = f'{DRIVE_ROOT}/checkpoints'


In [ ]:
#Clone or Pull Repo
import pathlib

REPO_URL = "https://github.com/mehulbhardwaj/emotion-classifier-capstone.git"
REPO_DIR = "/content/emotion-classifier-capstone"

if pathlib.Path(REPO_DIR).exists():
    %cd $REPO_DIR
    !git pull --quiet
else:
    !git clone $REPO_URL $REPO_DIR --quiet
    %cd $REPO_DIR


In [ ]:
#Cell 3 – Install dependencies via pip
!pip install -U pip wheel
!pip install -q -r requirements.txt

# Cache HF models & datasets on Drive
import os
os.environ["HF_HOME"]        = f"{DRIVE_ROOT}/hf_cache"
os.environ["HF_DATASETS_CACHE"] = f"{DRIVE_ROOT}/hf_cache"


In [ ]:
#Cell 4 – Emit a session-specific YAML override
import pathlib, textwrap

cfg_text = textwrap.dedent(f"""
defaults:
  - _self_
  - model: mlp_fusion
  - trainer: colab_gpu

paths:
  data_root: {DATA_ROOT}
  ckpt_root: {CKPT_ROOT}

training:
  seed: 42
""")

path_cfg = pathlib.Path("configs/path_colab.yaml")
path_cfg.write_text(cfg_text)
print(path_cfg.read_text())


In [ ]:
#Cell 5 – Run (or resume) training
# ────────────────────────────────────────────────────────────────────────────
# This picks up `last.ckpt` (if present), writes new ckpts every 500 steps,
# and now cfg.experiment_name is guaranteed to exist.
# ────────────────────────────────────────────────────────────────────────────
!python main.py \
   --config_file configs/path_colab.yaml \
   --train_model \
   --num_epochs 1 \
   --limit_dialogues_train 50 \
   --limit_dialogues_dev 10 \
   --limit_dialogues_test 10 \
   --batch_size 16 \
   --learning_rate 3e-4 \
   --experiment_name mlp_fusion


In [ ]:
#Cell 6 – Validation / Inference
!python main.py \
   --config_file configs/path_colab.yaml \
   --evaluate_model {CKPT_ROOT}/mlp_fusion/last.ckpt
